# ChurnIQ — Telecom Customer Churn & Revenue Risk Analyzer

## Phase 6.4 — XGBoost Modeling

### Objective

Train an XGBoost classifier using the final leakage-safe dataset and compare its performance against the Random Forest benchmark.

### Current Benchmark

Random Forest

- ROC-AUC: 0.659
- Recall: 81.88% (Threshold = 0.25)
- F1 Score: 0.4908

### Goal

Identify the strongest predictive model for customer churn.

In [4]:
pip install xgboost

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)
Note: you may need to restart the kernel to use updated packages.


In [5]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [6]:
# ==========================================
# LOAD DATASET
# ==========================================

df = pd.read_csv("../data/processed/feature_engineered.csv")

print(df.shape)

(51047, 67)


C:\Users\Akshit\AppData\Local\Temp\ipykernel_21076\1374272833.py:5: DtypeWarning: Columns (66) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/feature_engineered.csv")


In [7]:
# ==========================================
# TARGET ENCODING
# ==========================================

df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [8]:
# ==========================================
# FINAL FEATURE SELECTION
# ==========================================

columns_to_drop = [
    "CustomerID",
    "ServiceArea",
    "MarketCode",
    "AreaCode",
    "CreditRating",
    "RetentionCalls",
    "RetentionOffersAccepted",
    "MadeCallToRetentionTeam"
]

model_df = df.drop(columns=columns_to_drop)

print(model_df.shape)

(51047, 59)


In [9]:
X = model_df.drop(columns=["Churn"])
y = model_df["Churn"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [12]:
numerical_features = X_train.select_dtypes(
    exclude=["object"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical:", len(numerical_features))
print("Categorical:", len(categorical_features))

Numerical: 38
Categorical: 20


In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

print("Preprocessor ready.")

Preprocessor ready.


In [14]:
# ==========================================
# CLASS IMBALANCE RATIO
# ==========================================

negative_class = (y_train == 0).sum()
positive_class = (y_train == 1).sum()

scale_pos_weight = negative_class / positive_class

print("Negative Class:", negative_class)
print("Positive Class:", positive_class)
print("Scale Pos Weight:", round(scale_pos_weight, 2))

Negative Class: 29068
Positive Class: 11769
Scale Pos Weight: 2.47


## Step 2: Train Baseline XGBoost

A baseline XGBoost model is trained using:

- Leakage-safe feature set
- OneHot encoding pipeline
- Class imbalance adjustment

This model will be compared against the Random Forest benchmark.

In [15]:
# ==========================================
# XGBOOST PIPELINE
# ==========================================

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=300,
                max_depth=5,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=scale_pos_weight,
                random_state=42,
                eval_metric="logloss"
            )
        )
    ]
)

print("XGBoost pipeline created.")

XGBoost pipeline created.


In [16]:
# ==========================================
# TRAIN XGBOOST
# ==========================================

xgb_pipeline.fit(
    X_train,
    y_train
)

print("XGBoost training completed.")

XGBoost training completed.


In [17]:
# ==========================================
# PREDICTIONS
# ==========================================

y_pred_xgb = xgb_pipeline.predict(
    X_valid
)

y_prob_xgb = xgb_pipeline.predict_proba(
    X_valid
)[:, 1]

print("Predictions generated.")

Predictions generated.


In [18]:
# ==========================================
# XGBOOST METRICS
# ==========================================

accuracy_xgb = accuracy_score(
    y_valid,
    y_pred_xgb
)

precision_xgb = precision_score(
    y_valid,
    y_pred_xgb
)

recall_xgb = recall_score(
    y_valid,
    y_pred_xgb
)

f1_xgb = f1_score(
    y_valid,
    y_pred_xgb
)

roc_auc_xgb = roc_auc_score(
    y_valid,
    y_prob_xgb
)

print("Accuracy :", round(accuracy_xgb, 4))
print("Precision:", round(precision_xgb, 4))
print("Recall   :", round(recall_xgb, 4))
print("F1 Score :", round(f1_xgb, 4))
print("ROC AUC  :", round(roc_auc_xgb, 4))

Accuracy : 0.6253
Precision: 0.403
Recall   : 0.6241
F1 Score : 0.4897
ROC AUC  : 0.6767


## Step 3: XGBoost Threshold Tuning

The default classification threshold of 0.50 may not be optimal for churn prediction.

Multiple thresholds will be evaluated to identify the best trade-off between:

- Precision
- Recall
- F1 Score

The selected threshold will be used for final model comparison.

In [23]:
# ==========================================
# XGBOOST THRESHOLD TUNING
# ==========================================

thresholds = [0.50, 0.40, 0.35, 0.30, 0.25, 0.20]

results = []

for threshold in thresholds:

    y_pred_threshold = (
        y_prob_xgb >= threshold
    ).astype(int)

    precision = precision_score(
        y_valid,
        y_pred_threshold
    )

    recall = recall_score(
        y_valid,
        y_pred_threshold
    )

    f1 = f1_score(
        y_valid,
        y_pred_threshold
    )

    results.append([
        threshold,
        round(precision, 4),
        round(recall, 4),
        round(f1, 4)
    ])

xgb_threshold_df = pd.DataFrame(
    results,
    columns=[
        "Threshold",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

xgb_threshold_df

,Threshold,Precision,Recall,F1 Score
0,0.50,0.4030,0.6241,0.4897
1,0.40,0.3535,0.8266,0.4952
2,0.35,0.3361,0.8858,0.4873
3,0.30,0.3222,0.9286,0.4784
4,0.25,0.3115,0.9660,0.4711
5,0.20,0.2984,0.9847,0.4580


In [26]:
import joblib

joblib.dump(
    xgb_pipeline,
    "../models/champion_xgboost.joblib"
)

print("Champion model saved successfully.")

Champion model saved successfully.


In [27]:
import joblib

loaded_model = joblib.load(
    "../models/champion_xgboost.joblib"
)

type(loaded_model)

sklearn.pipeline.Pipeline

In [28]:
import json

metrics = {
    "roc_auc": 0.6800,
    "cv_std": 0.0044,
    "precision": 0.3535,
    "recall": 0.8266,
    "f1": 0.4952,
    "threshold": 0.40
}

with open(
    "../models/model_metrics.json",
    "w"
) as f:
    json.dump(
        metrics,
        f,
        indent=4
    )

print("Model metrics saved successfully.")

Model metrics saved successfully.
